# M6 — Réconciliation automatique des transactions

**Orange Money | Koceila SALEM**

Apparier automatiquement les transactions non réconciliées (rollbacks, corrections,
P2P en attente) qui encombrent le traitement manuel des équipes RA&FM.

## Démarche
1. **DIAGNOSTIC** : comprendre comment les colonnes de réconciliation sont liées
2. Choix de la méthode d'appariement (après diagnostic)
3. Appariement + score de confiance
4. Taux de réconciliation auto + cas résiduels

**Léger** : seulement ~0.3% des transactions sont concernées (~3000 cas).

## 0. Imports

In [ ]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path
ROOT = Path.cwd()
while not (ROOT / 'src').exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import joblib, json, time, warnings
warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-whitegrid')

from src import config as cfg
from src.data_loader import load_parquet

MODEL_DIR  = cfg.MODELS_DIR / 'M6_reconciliation'
OUTPUT_DIR = cfg.OUTPUTS_DIR / 'M6_reconciliation'
MODEL_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print('Imports OK')

## 1. Chargement des colonnes de réconciliation

In [ ]:
COLS_M6 = [cfg.COL_TRANSFER_ID, cfg.COL_DATE, cfg.COL_SENDER_ID, cfg.COL_RECVR_ID,
           cfg.COL_MONTANT, cfg.COL_SERVICE, cfg.COL_SUBTYPE, cfg.COL_STATUT,
           cfg.COL_ACTION, cfg.COL_TAG,
           cfg.COL_RECON_BY, cfg.COL_RECON_FOR,
           cfg.COL_ORIG_REF, cfg.COL_EXT_TXN, cfg.COL_REF_NUM]
COLS_M6 = list(dict.fromkeys(COLS_M6))
df = load_parquet(columns=COLS_M6)
df[cfg.COL_MONTANT] = pd.to_numeric(df[cfg.COL_MONTANT], errors='coerce').fillna(0)
print(f'Chargé : {len(df):,} transactions')

## 2. DIAGNOSTIC — comment les colonnes de réconciliation sont remplies

In [ ]:
print('=== REMPLISSAGE DES COLONNES DE RÉCONCILIATION ===\n')
for c in [cfg.COL_RECON_BY, cfg.COL_RECON_FOR, cfg.COL_ORIG_REF,
          cfg.COL_EXT_TXN, cfg.COL_REF_NUM, cfg.COL_ACTION]:
    non_null = df[c].notna().sum()
    pct = non_null / len(df) * 100
    nunique = df[c].nunique()
    print(f'{c:<24} : {non_null:>8,} non-null ({pct:5.2f}%) | {nunique:,} valeurs uniques')

print('\n=== ACTION_TYPE (types d opération) ===')
print(df[cfg.COL_ACTION].value_counts(dropna=False).to_string())

## 3. DIAGNOSTIC — structure des liens de réconciliation

On cherche à comprendre : RECONCILIATION_FOR pointe-t-il vers un TRANSFER_ID existant ?

In [ ]:
# Transactions AVEC une info de réconciliation
a_reconcilier = df[df[cfg.COL_RECON_BY].notna() | df[cfg.COL_RECON_FOR].notna()].copy()
print(f'Transactions à réconcilier : {len(a_reconcilier):,}')

# Exemples concrets
print('\n=== EXEMPLES (5 premières) ===')
cols_show = [cfg.COL_TRANSFER_ID, cfg.COL_ACTION, cfg.COL_MONTANT,
            cfg.COL_RECON_BY, cfg.COL_RECON_FOR, cfg.COL_ORIG_REF]
cols_show = [c for c in cols_show if c in a_reconcilier.columns]
print(a_reconcilier[cols_show].head(5).to_string())

# Est-ce que RECONCILIATION_FOR pointe vers un TRANSFER_ID existant ?
ids_existants = set(df[cfg.COL_TRANSFER_ID].dropna())
recon_for_vals = a_reconcilier[cfg.COL_RECON_FOR].dropna()
matchs = recon_for_vals.isin(ids_existants).sum()
print(f'\nRECONCILIATION_FOR pointe vers un TRANSFER_ID existant : {matchs}/{len(recon_for_vals)} ({matchs/max(len(recon_for_vals),1)*100:.1f}%)')

# Idem pour ORIGINAL_REF_NUMBER
orig_vals = a_reconcilier[cfg.COL_ORIG_REF].dropna()
if len(orig_vals) > 0:
    m2 = orig_vals.isin(ids_existants).sum()
    print(f'ORIGINAL_REF_NUMBER pointe vers un TRANSFER_ID existant : {m2}/{len(orig_vals)} ({m2/len(orig_vals)*100:.1f}%)')

## 4. DIAGNOSTIC — lien avec ACTION_TYPE (rollback/correction)

In [ ]:
# Croiser ACTION_TYPE avec la présence de réconciliation
df['a_recon'] = (df[cfg.COL_RECON_BY].notna() | df[cfg.COL_RECON_FOR].notna())
print('Réconciliation par ACTION_TYPE :')
print(df.groupby(cfg.COL_ACTION, observed=True)['a_recon'].agg(['sum','mean','count']).to_string())

# Distribution des montants à réconcilier
print('\nMontants des transactions à réconcilier :')
print(a_reconcilier[cfg.COL_MONTANT].describe().round(0).to_string())

print('\n>>> ENVOIE CETTE SORTIE (cellules 2,3,4) pour choisir la méthode d appariement <<<')

## ─────────────────────────────────────────
## PARTIE 2 — APPARIEMENT (méthode validée par diagnostic)
## ─────────────────────────────────────────

**Diagnostic confirmé :** RECONCILIATION_FOR pointe vers un TRANSFER_ID existant à 99.1%.
On a donc une VÉRITÉ TERRAIN. Méthode en 3 niveaux :
1. Appariement DIRECT par clé (RECONCILIATION_FOR ↔ TRANSFER_ID)
2. Validation par RÈGLES (montant identique, délai court) → confiance
3. TF-IDF pour les ORPHELINS (lien direct manquant)

## 5. Niveau 1 — Appariement direct par clé

In [ ]:
# Index des transactions par TRANSFER_ID (dédoublonné : 1 ligne par ID)
# ACTION_TYPE MODIFICATION/APPROBATION crée des doublons de TRANSFER_ID
df_unique = df.drop_duplicates(subset=[cfg.COL_TRANSFER_ID], keep='first')
df_idx = df_unique.set_index(cfg.COL_TRANSFER_ID)
ids_existants = set(df_unique[cfg.COL_TRANSFER_ID].dropna())
print(f'TRANSFER_ID uniques : {len(ids_existants):,} (sur {len(df):,} lignes)')

# Transactions à réconcilier via RECONCILIATION_FOR
a_rec = df[df[cfg.COL_RECON_FOR].notna()].copy()
print(f'Transactions avec RECONCILIATION_FOR : {len(a_rec):,}')

# Niveau 1 : le lien pointe-t-il vers un TRANSFER_ID existant ?
a_rec['jumeau_id'] = a_rec[cfg.COL_RECON_FOR]
a_rec['match_direct'] = a_rec['jumeau_id'].isin(ids_existants)

n_direct = a_rec['match_direct'].sum()
print(f'Appariés directement : {n_direct:,} ({n_direct/len(a_rec)*100:.1f}%)')
print(f'Orphelins (lien cassé) : {(~a_rec["match_direct"]).sum():,}')

## 6. Niveau 2 — Validation par règles (score de confiance)

In [ ]:
# Pour les appariés directs, vérifier la cohérence montant + temps
matches = a_rec[a_rec['match_direct']].copy()

# Maps depuis l'index dédoublonné (sûr car df_idx a un index unique)
map_montant = df_idx[cfg.COL_MONTANT]
map_date    = df_idx[cfg.COL_DATE]
matches['jumeau_montant'] = matches['jumeau_id'].map(map_montant)
matches['jumeau_date']    = matches['jumeau_id'].map(map_date)

# Règle 1 : montant identique (au centime près)
matches['regle_montant'] = (np.abs(matches[cfg.COL_MONTANT] - matches['jumeau_montant']) < 1).astype(int)
# Règle 2 : délai court (< 24h)
delai = (matches[cfg.COL_DATE] - matches['jumeau_date']).abs().dt.total_seconds() / 3600
matches['regle_temps'] = (delai < 24).fillna(False).astype(int)

# Score de confiance : 50 (lien direct) + 25 (montant) + 25 (temps)
matches['confiance'] = 50 + matches['regle_montant']*25 + matches['regle_temps']*25

print('Distribution de la confiance :')
print(matches['confiance'].value_counts().sort_index().to_string())
print(f'\nMontant identique : {matches["regle_montant"].mean()*100:.1f}%')
print(f'Délai < 24h       : {matches["regle_temps"].mean()*100:.1f}%')
print(f'\nAppariements haute confiance (100) : {(matches["confiance"]==100).sum():,}')

## 7. Niveau 3 — TF-IDF pour les orphelins

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

orphelins = a_rec[~a_rec['match_direct']].copy()
print(f'Orphelins à traiter par TF-IDF : {len(orphelins):,}')

if len(orphelins) > 0:
    # Candidats jumeaux : transactions de même montant (réduit l'espace de recherche)
    montants_orph = set(orphelins[cfg.COL_MONTANT].round(0))
    candidats = df[df[cfg.COL_MONTANT].round(0).isin(montants_orph)].copy()
    print(f'Candidats potentiels (même montant) : {len(candidats):,}')

    # TF-IDF sur les références (caractères) pour matcher des IDs similaires
    # On compare la référence de l'orphelin aux TRANSFER_ID candidats
    def cle_texte(s):
        return s.fillna('').astype(str)

    # Si peu d'orphelins, matching par montant+temps proche suffit
    appar_orph = []
    for _, row in orphelins.iterrows():
        m = row[cfg.COL_MONTANT]
        t = row[cfg.COL_DATE]
        # candidats même montant, délai < 24h, ID différent
        c = candidats[(np.abs(candidats[cfg.COL_MONTANT]-m)<1) &
                      (candidats[cfg.COL_TRANSFER_ID]!=row[cfg.COL_TRANSFER_ID])]
        if len(c) > 0:
            c = c.copy()
            c['delai'] = (c[cfg.COL_DATE]-t).abs().dt.total_seconds()/3600
            best = c.nsmallest(1, 'delai')
            if best['delai'].iloc[0] < 24:
                appar_orph.append({'TRANSFER_ID':row[cfg.COL_TRANSFER_ID],
                                   'jumeau_id':best[cfg.COL_TRANSFER_ID].iloc[0],
                                   'confiance':40})  # confiance plus basse (déduit)
    print(f'Orphelins récupérés : {len(appar_orph):,}')
else:
    appar_orph = []
    print('Aucun orphelin')

## 8. Métriques de réconciliation

In [ ]:
n_total = len(a_rec)
n_apparies = n_direct + len(appar_orph)
taux_auto = n_apparies / n_total * 100

print('=== MÉTRIQUES M6 ===')
print(f'Transactions à réconcilier : {n_total:,}')
print(f'Appariées automatiquement  : {n_apparies:,} ({taux_auto:.1f}%)')
print(f'  - lien direct  : {n_direct:,}')
print(f'  - via règles   : {len(appar_orph):,}')
print(f'Cas résiduels (analyste)   : {n_total - n_apparies:,}')
print(f'\nConfiance moyenne : {matches["confiance"].mean():.0f}/100')
print(f'Haute confiance (=100)     : {(matches["confiance"]==100).sum():,} ({(matches["confiance"]==100).mean()*100:.1f}%)')

# Estimation gain de temps analyste (ex: 3 min par cas manuel évité)
gain_heures = n_apparies * 3 / 60
print(f'\nGain estimé : {gain_heures:,.0f} heures analyste économisées (3 min/cas)')

## 9. Export

In [ ]:
# Résultats appariés
result = matches[[cfg.COL_TRANSFER_ID,'jumeau_id',cfg.COL_MONTANT,cfg.COL_ACTION,
                  'regle_montant','regle_temps','confiance']].copy()
result['methode'] = 'direct'
result.to_parquet(OUTPUT_DIR/'scored.parquet', index=False)
result.to_csv(OUTPUT_DIR/'appariements.csv', index=False, encoding='utf-8-sig')

# Cas résiduels
ids_apparies = set(matches[cfg.COL_TRANSFER_ID]) | set(o['TRANSFER_ID'] for o in appar_orph)
residuels = a_rec[~a_rec[cfg.COL_TRANSFER_ID].isin(ids_apparies)]
residuels[[cfg.COL_TRANSFER_ID,cfg.COL_MONTANT,cfg.COL_RECON_FOR]]\
    .to_csv(OUTPUT_DIR/'cas_residuels.csv', index=False, encoding='utf-8-sig')

params = {
    'methode':'direct + regles + tfidf orphelins',
    'taux_auto': float(taux_auto),
    'n_total': int(n_total), 'n_apparies': int(n_apparies),
    'regle_montant_seuil': 1, 'regle_temps_heures': 24,
}
with open(MODEL_DIR/'params.json','w') as f:
    json.dump(params, f, indent=2, default=str)

print('Exports créés dans outputs/M6_reconciliation/')
print(f'\n=== RÉSUMÉ M6 ===')
print(f'Taux réconciliation auto : {taux_auto:.1f}%')
print(f'Appariements : {n_apparies:,}')
print(f'Cas résiduels : {n_total-n_apparies:,}')